In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import re
import sys

from partd import python
from pathlib import Path
import gc

sys.path.append(os.path.abspath('../../'))
from utils_mitgcm import *
from utils_modal_analysis import load_mode_cache,  process_mode, prepare_vector_mode

# Load MITgcm results

In [2]:
lake = 'geneva'
model = f'{lake}_2025'

In [ ]:
mitgcm_config, ds_mitgcm = open_mitgcm_ds_from_config('../../config.json', model)
base_folder_path = os.path.dirname(mitgcm_config['datapath'])

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f357e03a550>>
Traceback (most recent call last):
  File "/home/leroquan@eawag.wroot.emp-eaw.ch/miniconda3/envs/horizontal_structures/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
horizontal_resolution = 100
ds_mitgcm['YG'] = np.arange(0, len(ds_mitgcm['YG'])) * horizontal_resolution
ds_mitgcm['XG'] = np.arange(0, len(ds_mitgcm['XG'])) * horizontal_resolution
ds_mitgcm['YC'] = np.arange(1, len(ds_mitgcm['YC']) + 1) * horizontal_resolution - horizontal_resolution / 2
ds_mitgcm['XC'] = np.arange(1, len(ds_mitgcm['XC']) + 1) * horizontal_resolution - horizontal_resolution / 2

# Get folder & files

In [ ]:
base_folder = rf"/storage/alplakes_test/{lake}_100m_2025"

In [ ]:
modal_analysis_dir = os.path.join(base_folder, "modal_analysis")

In [ ]:
mode_date_str = "2025-08-07"
date_dir = Path(modal_analysis_dir) / mode_date_str
files = list(date_dir.rglob("*mode*.nc"))

# Process each layer

In [ ]:
mode_cache = load_mode_cache(files)

In [10]:
prepared_mode = [
    prepare_vector_mode(
        ds_mitgcm.UVEL,
        ds_mitgcm.VVEL,
        ds_mode.u1,
        ds_mode.v1,
        dA=horizontal_resolution**2,
        rho=1025.0,
        normalize_mode=False
    )
    for name, ds_mode in mode_cache]

In [11]:
xr_norm2 = xr.DataArray(prepared_mode[0]['norm2'],
                        dims=['Z'],
                        coords={'Z': ds_mitgcm.Z},
                        name='norm2')
xr_norm2.to_netcdf(os.path.join(modal_analysis_dir, "norm2.nc"))

In [ ]:
import psutil

In [ ]:
for month in range(4,12): 
    for z_range in [range(0,10), range(10,20), range(20,30), range(30,40), range(40,50)]: 
        print(psutil.virtual_memory())
        ds_crop = ds_mitgcm.isel(Z=z_range).sel(
            time=slice(
                pd.to_datetime(f"2025-{month:02d}-01"),
                pd.to_datetime(f"2025-{month+1:02d}-01")
            )
        )

        ds_crop = ds_crop.load() #.chunk({'time': 100, 'Z': 1})

        print(f'Finished loading month {month}.')

        results = [
            process_mode(
                name,
                ds_mode,
                ds_crop.UVEL,
                ds_crop.VVEL,
                horizontal_resolution**2,
            )
            for name, ds_mode in mode_cache
        ]

        ds_modes = xr.concat(results, dim="mode").compute()
        
        print(f'Saving month {month:02d}, Z range {z_range[0]}-{z_range[-1]}...')
        ds_modes.to_netcdf(os.path.join(modal_analysis_dir, f"KE_projected_optimized_month{month:02d}_Z{z_range[0]}-{z_range[-1]}.nc")) 
        print(f'Month {month}, Z range {z_range[0]}-{z_range[-1]} is done.')
        del ds_modes
        del ds_crop
        gc.collect()

svmem(total=270322470912, available=184465285120, percent=31.8, used=83537080320, free=171782471680, active=74226249728, inactive=20040175616, buffers=107737088, cached=14895181824, shared=428331008, slab=2705027072)


In [ ]:
(ds_modes.isel(mode=0).sum(dim='Z').KE/1e6).plot()

In [ ]:
ds_test = xr.open_dataset(os.path.join(modal_analysis_dir, f"KE_projected.nc"))

In [ ]:
del ds_modes
del ds_crop
gc.collect()

In [ ]:
from glob import glob
import os
import xarray as xr

files = [
    sorted(glob(os.path.join(modal_analysis_dir, "KE_projected_optimized_month*_Z0-9.nc"))),
    sorted(glob(os.path.join(modal_analysis_dir, "KE_projected_optimized_month*_Z10-19.nc"))),
    sorted(glob(os.path.join(modal_analysis_dir, "KE_projected_optimized_month*_Z20-29.nc"))),
    sorted(glob(os.path.join(modal_analysis_dir, "KE_projected_optimized_month*_Z30-39.nc"))),
    sorted(glob(os.path.join(modal_analysis_dir, "KE_projected_optimized_month*_Z40-49.nc"))),
]

ds_test = xr.open_mfdataset(
    files,
    combine="nested",
    concat_dim=["Z", "time"]
)

In [ ]:
ds_test.U_real.sum(dim=['mode']).isel(Z=0).isel(time=50).plot()

In [ ]:
plt.figure(figsize=(20,5))
ds_test.isel(mode=0).KE.sum(dim=['Z']).plot(label='Mode 0')
ds_test.isel(mode=1).KE.sum(dim=['Z']).plot(label='Mode 1')
#ds_test.isel(mode=2).KE.sum(dim=['Z']).plot(label='Mode 2')
plt.legend()

In [ ]:
(ds_test.KE.sum(dim=['mode', 'Z'])/1e6).to_dataframe(name='kinetic_energy_[MJ]').to_csv(os.path.join(modal_analysis_dir, 'KE_projected.csv'))

In [ ]:
ds_test.drop_vars(["A_real", "A_imag", "U_imag", "V_imag"]).isel(mode=[0,2]).chunk().to_netcdf(os.path.join(modal_analysis_dir, f"KE_projected_optimized_Z0-10.nc")) 

In [ ]:
test = 0

ds_proj_concat.to_netcdf(os.path.join(modal_analysis_dir, f"KE_projected_optimized_0-5.nc")) 

from utils_modal_analysis import _match_mode_to_field

def prepare_vector_mode(
    U,
    V,
    mode_u,
    mode_v,
    dA,
    rho=1025.0,
    normalize_mode=True,
):
    """
    Precompute all mode-dependent quantities, including Z-dependent weights.
    """

    mode_u = _match_mode_to_field(U.isel(Z=0), mode_u)
    mode_v = _match_mode_to_field(V.isel(Z=0), mode_v)

    # Z-dependent weights
    w_u = np.sqrt(rho * U["drF"] * dA)
    w_v = np.sqrt(rho * V["drF"] * dA)

    # Weighted modes
    mode_uw = mode_u * w_u
    mode_vw = mode_v * w_v

    mode_uw_np = []
    mode_vw_np = []
    norm2 = []
    
    def get_norm2(mode_uw_np_i, mode_vw_np_i):
        # Mode norm
        norm2_i = (
            np.vdot(mode_uw_np_i, mode_uw_np_i).real
            +
            np.vdot(mode_vw_np_i, mode_vw_np_i).real
        )

        return norm2_i

    for zz in range(len(w_u)):
        mode_uw_np_i = mode_uw.isel(Z=zz).values.ravel()
        mode_uw_np.append(mode_uw_np_i)

        mode_vw_np_i = mode_vw.isel(Z=zz).values.ravel()
        mode_vw_np.append(mode_vw_np_i)

        norm2.append(
            get_norm2(
                mode_uw_np_i, 
                mode_vw_np_i))

    norm = np.sqrt(norm2)

    if normalize_mode:

        norm = np.sqrt(norm2)

        mode_u = mode_u.expand_dims({"Z": U["Z"]}) / xr.DataArray(norm, dims=["Z"])
        mode_v = mode_v.expand_dims({"Z": U["Z"]}) / xr.DataArray(norm, dims=["Z"])

        mode_uw_np /= norm[:, None]
        mode_vw_np /= norm[:, None]

        norm2 = np.ones_like(norm2)

    return {
        "mode_u": mode_u,
        "mode_v": mode_v,

        "mode_uw_np": mode_uw_np,
        "mode_vw_np": mode_vw_np,

        "norm2": norm2,

        "w_u": w_u,
        "w_v": w_v,
    }

prepared_mode = prepare_vector_mode(
    ds_crop.UVEL,
    ds_crop.VVEL,
    mode_cache[0][1].u1,
    mode_cache[0][1].v1,
    dA=horizontal_resolution**2,
    rho=1025.0,
)

prepared_mode['mode_u'].isel(Z=49).real.plot()

depth_suffix = '_-0.25--56.6m'
seiche_ke = pd.read_csv(os.path.join(base_folder_path, "energy_budget", f"ke_seiche{depth_suffix}.csv"))
seiche_ke['date'] = pd.to_datetime(seiche_ke['time'])
seiche_ke = seiche_ke.set_index('date')['kinetic_energy_[MJ]']

KE_opti = ds_proj_concat.KE.sum(dim=['mode', 'Z'])/1e6

seiche_ke.plot()
KE_opti.plot()
plt.xlim(None, pd.to_datetime("2025-06-01"))